# Optimized neural CFR+ 45-minute reference

This run promotes the low-fitting screen winner to a full reference configuration:

- 1024 traversals per player;
- 24 regret optimizer steps;
- 6 average-strategy optimizer steps;
- 2048x2048 regret network and 512x512 strategy network.

Measured training time excludes asynchronous exact exploitability, policy serialization, and checkpointing. Comparison cells automatically discover the completed Deep CFR and older CFR+ references. A neural FSP screen is included if present, but is labelled as a screen rather than a reference run.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.eval.neural_evaluators import (
    AsyncExactExploitabilityEvaluator,
    ScheduledEvaluation,
)
from liars_poker.training.neural_runs import run_neural_cfr_plus

assert torch.cuda.is_available(), 'This reference run requires CUDA.'
device = 'cuda'
torch.set_float32_matmul_precision('high')
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips'),
    suit_symmetry=True,
)

TRAINING_MINUTES = 45
EVALUATE_EVERY_MINUTES = 5
CHECKPOINT_EVERY_MINUTES = 15
TRAVERSALS_PER_PLAYER = 1024
SEED = 7
RUN_REFERENCE = True
SAVE_RESUME_CHECKPOINT = True
DELETE_EVALUATION_SNAPSHOTS = True

TRAINER_KWARGS = {
    'regret_hidden_sizes': (2048, 2048),
    'strategy_hidden_sizes': (512, 512),
    'device': device,
    'seed': SEED,
    'regret_buffer_capacity': 500_000,
    'strategy_buffer_capacity': 2_000_000,
    'learning_rate': 1e-3,
    'batch_size': 4096,
    'regret_train_steps': 24,
    'strategy_train_steps': 6,
    'regret_positive_weight': 0.5,
    'strategy_weighting': 'linear',
    'validation_fraction': 0.02,
    'validation_buffer_capacity': 20_000,
    'traversal_backend': 'gpu_native',
    'traversal_batch_size': 1024,
    'device_replay': True,
    'fused_optimizer': True,
    'amp_dtype': None,
    'compile_models': False,
}

run_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
RUN_DIR = (
    REPO_ROOT / 'artifacts' / 'deep_cfr_plus_reference_runs'
    / f'{spec.to_short_str()}___optimized_24r6s___{run_id}'
)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('run directory:', RUN_DIR)

In [ ]:
def exact_schedule():
    return ScheduledEvaluation(
        name='exact_exploitability',
        evaluator=AsyncExactExploitabilityEvaluator(
            max_workers=1,
            compile_batch_size=65_536,
        ),
        every_minutes=EVALUATE_EVERY_MINUTES,
        run_at_end=True,
    )


def mean_validation(metrics, section, metric):
    values = [row[metric] for row in metrics[section] if row.get('records', 0)]
    return float(np.mean(values)) if values else np.nan


def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    return [
        json.loads(line)
        for line in path.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]


def normalized_auc(frame, x='measured_training_s', y='exploitability'):
    ordered = frame.sort_values(x)
    xs = ordered[x].to_numpy(dtype=float)
    ys = ordered[y].to_numpy(dtype=float)
    if len(xs) < 2 or xs[-1] <= xs[0]:
        return np.nan
    return float(np.trapezoid(ys, xs) / (xs[-1] - xs[0]))

In [ ]:
reference_result = None

if RUN_REFERENCE:
    if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
        raise FileExistsError(
            f'{RUN_DIR} already contains data. Rerun the configuration cell '
            'to generate a new run ID, or resume it explicitly with the API.'
        )
    reference_result = run_neural_cfr_plus(
        spec,
        minutes=TRAINING_MINUTES,
        traversals_per_player=TRAVERSALS_PER_PLAYER,
        trainer_kwargs=TRAINER_KWARGS,
        evaluations=[exact_schedule()],
        checkpoint_every_minutes=(
            CHECKPOINT_EVERY_MINUTES if SAVE_RESUME_CHECKPOINT else None
        ),
        run_dir=RUN_DIR,
        save_checkpoint=SAVE_RESUME_CHECKPOINT,
        wait_for_evaluations=True,
        debug=True,
    )

    validation = reference_result.trainer.validation_metrics()
    final_diagnostics = pd.DataFrame([{
        'iterations': reference_result.trainer.iteration,
        'measured training min': reference_result.measured_training_s / 60.0,
        'wall min': reference_result.wall_s / 60.0,
        'final regret MSE': mean_validation(validation, 'regret', 'mse'),
        'final regret strategy TV': mean_validation(validation, 'regret', 'strategy_tv'),
        'final average-network TV': mean_validation(validation, 'strategy', 'strategy_tv'),
    }])
    final_diagnostics.to_csv(RUN_DIR / 'final_diagnostics.csv', index=False)
    display(final_diagnostics.style.format(precision=6))

    if DELETE_EVALUATION_SNAPSHOTS:
        snapshots = RUN_DIR / 'evaluations' / 'exact_exploitability' / 'snapshots'
        if snapshots.exists():
            shutil.rmtree(snapshots)

    print('completed run:', reference_result.run_dir)

The checkpoint is deliberately retained until the run has been inspected. Once satisfied with the saved policy and logs, delete it with the optional cleanup cell at the bottom.

In [ ]:
def manifest_for(run_dir):
    path = Path(run_dir) / 'manifest.json'
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}


def matching_spec(manifest):
    stored = manifest.get('spec')
    return stored == spec.to_json()


def load_exact_reference(run_dir, label):
    run_dir = Path(run_dir)
    old_path = run_dir / 'exact_results.jsonl'
    new_path = run_dir / 'evaluations.jsonl'
    records = read_jsonl(new_path if new_path.exists() else old_path)
    frame = pd.DataFrame(records)
    if frame.empty:
        return frame
    if 'evaluator' in frame:
        frame = frame[frame['evaluator'] == 'exact_exploitability'].copy()
    if 'iteration' not in frame and 'iter' in frame:
        frame['iteration'] = frame['iter']
    frame['label'] = label
    frame['source'] = str(run_dir)
    return frame.sort_values('measured_training_s').drop_duplicates(
        subset='iteration', keep='last'
    )


def discover_reference_curves():
    curves = []

    deep_root = REPO_ROOT / 'artifacts' / 'deep_cfr_reference_runs'
    deep_candidates = []
    for run_dir in deep_root.glob('*') if deep_root.exists() else []:
        manifest = manifest_for(run_dir)
        if (
            manifest.get('status') == 'complete'
            and manifest.get('run_type') == 'deep_cfr_gpu_reference'
            and matching_spec(manifest)
        ):
            deep_candidates.append(run_dir)
    if deep_candidates:
        chosen = max(deep_candidates, key=lambda path: path.stat().st_mtime)
        curves.append(load_exact_reference(chosen, 'Deep CFR reference'))

    plus_root = REPO_ROOT / 'artifacts' / 'deep_cfr_plus_reference_runs'
    old_plus = []
    optimized_plus = []
    for run_dir in plus_root.glob('*') if plus_root.exists() else []:
        manifest = manifest_for(run_dir)
        if manifest.get('status') != 'complete' or not matching_spec(manifest):
            continue
        kwargs = manifest.get('trainer_kwargs', {})
        key = (
            int(kwargs.get('regret_train_steps', -1)),
            int(kwargs.get('strategy_train_steps', -1)),
        )
        if key == (100, 50):
            old_plus.append(run_dir)
        elif key == (24, 6):
            optimized_plus.append(run_dir)
    if old_plus:
        chosen = max(old_plus, key=lambda path: path.stat().st_mtime)
        curves.append(load_exact_reference(chosen, 'Neural CFR+ old reference (100/50)'))
    if optimized_plus:
        chosen = max(optimized_plus, key=lambda path: path.stat().st_mtime)
        curves.append(load_exact_reference(chosen, 'Neural CFR+ optimized reference (24/6)'))

    screen_root = REPO_ROOT / 'artifacts' / 'neural_method_screens'
    fsp_files = sorted(
        screen_root.glob('*/neural_fsp_*.csv'),
        key=lambda path: path.stat().st_mtime,
    ) if screen_root.exists() else []
    fsp_frames = []
    for path in fsp_files:
        frame = pd.read_csv(path)
        if frame.empty or 'configuration' not in frame:
            continue
        selected = frame[
            frame['configuration'].str.contains('frequent updates', case=False)
        ].copy()
        if not selected.empty:
            selected['measured_training_s'] = selected['measured training s']
            selected['iteration'] = selected['outer iteration']
            selected['exploitability'] = selected['learned average exploitability']
            selected['label'] = 'Neural FSP screen (50-BR updates)'
            selected['source'] = str(path)
            fsp_frames.append(selected)
    if fsp_frames:
        curves.append(fsp_frames[-1])

    return pd.concat([frame for frame in curves if not frame.empty], ignore_index=True)


comparison = discover_reference_curves()
display(
    comparison.groupby('label', sort=False)
    .agg(
        evaluations=('exploitability', 'size'),
        final_training_min=('measured_training_s', lambda x: x.iloc[-1] / 60.0),
        final_exploitability=('exploitability', 'last'),
        best_exploitability=('exploitability', 'min'),
    )
    .style.format(precision=6)
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.4))

for label, frame in comparison.groupby('label', sort=False):
    frame = frame.sort_values('measured_training_s')
    axes[0].plot(
        frame['measured_training_s'] / 60.0,
        frame['exploitability'],
        marker='o',
        label=label,
    )
    if 'Neural FSP' not in label:
        axes[1].plot(
            frame['iteration'],
            frame['exploitability'],
            marker='o',
            label=label,
        )

for ax, title, xlabel in [
    (axes[0], 'Reference comparison by measured training time', 'Measured neural-training minutes'),
    (axes[1], 'CFR-family comparison by iteration', 'Algorithm iteration'),
]:
    ax.set(title=title, xlabel=xlabel, ylabel='Exact exploitability')
    ax.set_yscale('log')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(fontsize=8)

fig.tight_layout()

In [ ]:
summary_rows = []
for label, frame in comparison.groupby('label', sort=False):
    frame = frame.sort_values('measured_training_s')
    summary_rows.append({
        'method': label,
        'training min represented': frame['measured_training_s'].iloc[-1] / 60.0,
        'iterations represented': int(frame['iteration'].iloc[-1]),
        'final exploitability': frame['exploitability'].iloc[-1],
        'best exploitability': frame['exploitability'].min(),
        'normalized AUC over own horizon': normalized_auc(frame),
    })

comparison_summary = pd.DataFrame(summary_rows).set_index('method')
display(comparison_summary.style.format(precision=6))
comparison_summary.to_csv(RUN_DIR / 'reference_comparison.csv')

The neural FSP curve covers only its short screen horizon, so its final value and own-horizon AUC are not directly comparable to 45-minute reference endpoints. It is included to show the order-of-magnitude performance difference, not as a matched-budget reference.

In [ ]:
# Optional storage cleanup after verifying the final policy and comparison.
# This removes exact-resume state but keeps the playable policy and all logs.
DELETE_RESUME_CHECKPOINT = False

if DELETE_RESUME_CHECKPOINT:
    checkpoint = RUN_DIR / 'latest_checkpoint.pt'
    if checkpoint.exists():
        checkpoint.unlink()
        print('removed:', checkpoint)
    else:
        print('no checkpoint present')